# D3 — GraphRAG Executor, Evaluation & Safety

This notebook demonstrates the complete D3 pipeline:

1. **Parameter Injection** — AutoML winning config loaded into hybrid search
2. **GraphRAG Pipeline** (Method A — Pre-Filter):
   - LLM generates Cypher → Neo4j subgraph → filtered search → LLM answer
3. **Provenance Filtering** — Safety check verifying citations against metadata
4. **Feedback Loop** — River online learning + ADWIN drift detection
5. **Evaluation & Ablation** — vector_only vs graph_only vs hybrid

**Prerequisites:**
- MongoDB, Qdrant, Neo4j running via `docker compose up -d mongodb qdrant neo4j`
- Data seeded via `python seed_data.py`
- `LLM_API_KEY` set in `.env` (or as environment variable)

## 0. Setup & Imports

In [1]:
import sys, os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Load .env if it exists
env_path = os.path.join(PROJECT_ROOT, ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ.setdefault(key.strip(), val.strip())
    print("Loaded .env file.")

print(f"Working directory: {os.getcwd()}")
print(f"LLM_MODEL: {os.getenv('LLM_MODEL', 'gpt-4o-mini')}")
print(f"LLM_API_KEY set: {'Yes' if os.getenv('LLM_API_KEY') else 'NO — set this before running!'}")

Loaded .env file.
Working directory: E:\PythonProject\PDF-Paper-AI-Agent
LLM_MODEL: gemini-2.5-flash
LLM_API_KEY set: Yes


---
## 1. Parameter Injection (Config to Search)

`hybrid_search.py` automatically loads the AutoML winning configuration from `configs/run_card.yaml` at import time. This means every search uses the optimized parameters without manual tuning.

In [2]:
import yaml

# Show the raw config
with open("configs/run_card.yaml") as f:
    config = yaml.safe_load(f)

print("AutoML Winning Config:")
for key, val in config["winning_config"].items():
    print(f"  {key}: {val}")

print("\nAutoML Best Metrics vs Baseline:")
for name, metrics in config["metrics"].items():
    print(f"  {name}:")
    for mk, mv in metrics.items():
        print(f"    {mk}: {mv}")

AutoML Winning Config:
  k: 11
  alpha: 0.1154
  svd_dim: 256
  norm: none
  metric: cosine

AutoML Best Metrics vs Baseline:
  baseline:
    NDCG@5: 0.9826
    Recall@5: 1.0
    p95_latency_ms: 2.23
  automl_best:
    NDCG@5: 0.9963
    Recall@5: 1.0
    p95_latency_ms: 1.89


In [3]:
from hybrid_search import HybridSearcher, CONFIG_K, CONFIG_ALPHA, WINNING_CONFIG

print("Values loaded into hybrid_search.py:")
print(f"  CONFIG_K     = {CONFIG_K}  (default top_k for search)")
print(f"  CONFIG_ALPHA = {CONFIG_ALPHA}  (BM25 fusion weight)")
print(f"  Full config  = {WINNING_CONFIG}")

# Demonstrate that search() uses CONFIG_K by default
searcher = HybridSearcher()
results = searcher.search("transformer architecture")
print(f"\nsearch('transformer architecture') returned {len(results)} results (default k={CONFIG_K})")

E:\PythonProject\PDF-Paper-AI-Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-21 19:16:27,262 | INFO | Loaded winning config from E:\PythonProject\PDF-Paper-AI-Agent\configs\run_card.yaml: k=11, alpha=0.1154


Values loaded into hybrid_search.py:
  CONFIG_K     = 11  (default top_k for search)
  CONFIG_ALPHA = 0.1154  (BM25 fusion weight)
  Full config  = {'k': 11, 'alpha': 0.1154, 'svd_dim': 256, 'norm': 'none', 'metric': 'cosine'}


2026-06-21 19:16:27,595 | INFO | Building BM25 index from MongoDB ...


2026-06-21 19:16:27,659 | INFO | BM25 index built with 872 documents.


...

2026-06-21 19:16:35,185 | INFO | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 38.50it/s]


2026-06-21 19:16:35,284 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"



search('transformer architecture') returned 11 results (default k=11)


---
## 2. GraphRAG Executor (Core Pipeline)

The `GraphRAGExecutor` implements Method A (Pre-Filter):

```
Query
  |
  +-> Step 1: LLM generates Cypher
  |            |
  |            v
  |        Step 2: Neo4j -> paper_ids
  |            |
  v            v
  Step 3: Filtered BM25 + Dense + RRF
  |
  v
  Step 4: LLM generates answer with citations
  |
  v
  Step 5: Provenance filter verifies citations
```

If the graph filter returns no results, it falls back to unfiltered search.

In [4]:
from graphrag_executor import GraphRAGExecutor

executor = GraphRAGExecutor()
print("GraphRAGExecutor initialized.")
print(f"  LLM model: {executor._model}")
print(f"  Temperature: {executor._temperature}")

2026-06-21 19:16:36,528 | INFO | Connected to Neo4j at bolt://localhost:7687


GraphRAGExecutor initialized.
  LLM model: gemini-2.5-flash
  Temperature: 0.0


### 2.1 Step 1 — Cypher Generation

The LLM receives the full Neo4j schema and produces a Cypher query that returns `paper_id` values.

In [5]:
test_query = "What papers discuss attention mechanisms in NLP?"

cypher, intent = executor._generate_cypher(test_query)
print(f"Query  : {test_query}")
print(f"Intent : {intent}")
print(f"Cypher : {cypher}")

2026-06-21 19:16:36,861 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


2026-06-21 19:16:39,143 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


2026-06-21 19:16:39,157 | INFO | Cypher generated | intent: Find papers related to 'Natural Language Processing' that mention 'attention mechanism' in their title or abstract.


2026-06-21 19:16:39,157 | INFO | Cypher: MATCH (p:Paper)-[:HAS_TOPIC]->(t:Topic) WHERE toLower(t.name) = 'natural language processing' AND (toLower(p.title) CONTAINS 'attention mechanism' OR toLower(p.abstract) CONTAINS 'attention mechanism') RETURN p.paper_id AS paper_id LIMIT 20


Query  : What papers discuss attention mechanisms in NLP?
Intent : Find papers related to 'Natural Language Processing' that mention 'attention mechanism' in their title or abstract.
Cypher : MATCH (p:Paper)-[:HAS_TOPIC]->(t:Topic) WHERE toLower(t.name) = 'natural language processing' AND (toLower(p.title) CONTAINS 'attention mechanism' OR toLower(p.abstract) CONTAINS 'attention mechanism') RETURN p.paper_id AS paper_id LIMIT 20


### 2.2 Step 2 — Neo4j Subgraph Extraction

In [6]:
if cypher:
    paper_ids = executor._execute_cypher(cypher)
    print(f"Neo4j returned {len(paper_ids)} paper(s):")
    for pid in paper_ids:
        print(f"  - {pid}")
else:
    paper_ids = []
    print("No Cypher generated (query had no graph-structural angle).")

Neo4j returned 0 paper(s):


### 2.3 Step 3 — Filtered Hybrid Search

In [7]:
chunks, fallback = executor._filtered_search(
    test_query,
    paper_ids if paper_ids else None,
    top_k=5,
)

print(f"Retrieved {len(chunks)} chunks (fallback={fallback})")
for i, c in enumerate(chunks, 1):
    print(f"  [{i}] {c.citation()}")
    print(f"      {c.text[:100]}...\n")

2026-06-21 19:16:39,230 | INFO | Building BM25 index from MongoDB ...


2026-06-21 19:16:39,280 | INFO | BM25 index built with 872 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 77.09it/s]


2026-06-21 19:16:39,307 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


Retrieved 5 chunks (fallback=False)
  [1] [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
       of sequential computation, however, remains.
Attention mechanisms have become an integral part of c...

  [2] [Language Understanding, Jacob Devlin, Ming-Wei Chang et al.] "BERT: Pre-training of Deep Bidirectional Transformers for" (pp. 11-11)
      njoon Seo, Aniruddha Kembhavi, Ali Farhadi, and
Hannaneh Hajishirzi. 2017. Bidirectional attention
ﬂ...

  [3] [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 4-4)
      tention is identical to our algorithm, except for the scaling factor
of
1
√dk . Additive attention c...

  [4] [reproduce the tables, figures in this paper solely for use in journa

### 2.4 Step 4 — LLM Answer Generation

The LLM is strictly instructed to cite every claim using `[Authors] "Title" (pp. X-Y)` format, using only the retrieved chunks.

In [8]:
if chunks:
    raw_answer = executor._generate_answer(test_query, chunks)
    print("Raw LLM Answer:")
    print("=" * 72)
    print(raw_answer)
    print("=" * 72)
else:
    print("No chunks to generate answer from.")
    raw_answer = ""

2026-06-21 19:16:44,468 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Raw LLM Answer:
Attention mechanisms are discussed in the following papers:

*   "Provided proper attribution is provided, Google hereby grants permission to" by [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] discusses attention mechanisms as an integral part of sequence modeling and transduction models, and proposes the Transformer model which relies on attention [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2). It also details dot-product attention and additive attention [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 4-4) and multi-head attention [reproduce the tables, figures in this paper solely for use in journalistic

---
## 3. Provenance Filtering (Safety)

The provenance filter extracts every citation from the LLM answer and cross-checks:
- Does the **paper title** match a retrieved chunk?
- Do the **page numbers** fall within the chunk's page range?

Unverified citations are replaced with `[CITATION REMOVED - unverified]`.

In [9]:
from graphrag_executor import provenance_filter

if raw_answer and chunks:
    filtered_answer, verified, dropped, score = provenance_filter(raw_answer, chunks)

    print(f"Provenance Score: {score:.2%}")
    print(f"Verified citations: {len(verified)}")
    print(f"Dropped citations:  {len(dropped)}")

    if verified:
        print("\nVerified:")
        for c in verified:
            print(f"  + {c}")

    if dropped:
        print("\nDropped (failed provenance):")
        for c in dropped:
            print(f"  X {c}")

    print("\nFiltered answer:")
    print("=" * 72)
    print(filtered_answer)
    print("=" * 72)
else:
    print("Skipping provenance demo (no answer generated).")

2026-06-21 19:16:44,479 | WARNING | Provenance filter dropped 1/5 citations.


Provenance Score: 80.00%
Verified citations: 4
Dropped citations:  1

Verified:
  + [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
  + [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 4-4)
  + [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 5-5)
  + [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 12-12)

Dropped (failed provenance):
  X [Jacob Devlin, Ming-Wei Chang et al.] "BERT: Pre-training of

### 3.1 Provenance filter — synthetic test

To show the filter catching a fabricated citation:

In [10]:
from hybrid_search import SearchResult

# Create fake chunks with known metadata
fake_chunks = [
    SearchResult(
        chunk_id="c1", paper_id="p1",
        title="Attention Is All You Need",
        text="The dominant sequence transduction models...",
        score=0.9, page_start=1, page_end=3,
        authors=["Vaswani", "Shazeer"], source="rrf",
    ),
]

# Answer with one real and one fabricated citation
test_answer = (
    'Transformers use self-attention [Vaswani, Shazeer] "Attention Is All You Need" (pp. 1-3). '
    'They also use convolutions [Doe, Smith] "A Paper That Does Not Exist" (pp. 10-15).'
)

filtered, verified, dropped, score = provenance_filter(test_answer, fake_chunks)

print(f"Provenance score: {score:.0%}")
print(f"Verified: {verified}")
print(f"Dropped:  {dropped}")
print(f"\nFiltered output:\n{filtered}")

2026-06-21 19:16:44,489 | WARNING | Provenance filter dropped 1/2 citations.


Provenance score: 50%
Verified: ['[Vaswani, Shazeer] "Attention Is All You Need" (pp. 1-3)']
Dropped:  ['[Doe, Smith] "A Paper That Does Not Exist" (pp. 10-15)']

Filtered output:
Transformers use self-attention [Vaswani, Shazeer] "Attention Is All You Need" (pp. 1-3). They also use convolutions [CITATION REMOVED - unverified].


---
## 4. Full End-to-End Pipeline

The `executor.query()` method runs all 5 steps in sequence and returns a `GraphRAGResponse`.

In [11]:
import time

query = "How does the attention mechanism work in transformers?"

t0 = time.perf_counter()
response = executor.query(query, top_k=5)
elapsed = (time.perf_counter() - t0) * 1000

print(f"Query: {query}")
print(f"Latency: {elapsed:.0f}ms")
print(f"\n{'='*72}")
print(f"Intent             : {response.intent}")
print(f"Cypher             : {response.cypher_generated}")
print(f"Graph papers found : {response.graph_papers_found}")
print(f"Filter applied     : {response.graph_filter_applied}")
print(f"Fallback used      : {response.fallback}")
print(f"Chunks used        : {response.chunks_used}")
print(f"Provenance score   : {response.provenance_score:.0%}")
print(f"Verified citations : {len(response.verified_citations)}")
print(f"Dropped citations  : {len(response.dropped_citations)}")
print(f"BM25 top score     : {response.bm25_top_score}")
print(f"Dense top score    : {response.dense_top_score}")
print(f"{'='*72}")
print(f"\nAnswer:\n{response.answer}")

2026-06-21 19:16:45,295 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


2026-06-21 19:16:45,302 | INFO | Cypher generated | intent: no_match


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 71.43it/s]


2026-06-21 19:16:45,337 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


2026-06-21 19:16:51,233 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Query: How does the attention mechanism work in transformers?
Latency: 6737ms

Intent             : no_match
Cypher             : None
Graph papers found : 0
Filter applied     : False
Fallback used      : False
Chunks used        : 5
Provenance score   : 100%
Verified citations : 6
Dropped citations  : 0
BM25 top score     : 0.0
Dense top score    : 0.0

Answer:
The attention mechanism in Transformers relies entirely on self-attention, also known as intra-attention [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2). This mechanism relates different positions of a single sequence to compute a representation of that sequence [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2). It allows for 

---
## 5. Feedback Loop (River + ADWIN)

After each query, the user can submit y/n helpfulness feedback. This synchronously:
1. Updates the River LogisticRegression model weights
2. Triggers ADWIN to check for concept drift in the error stream

The adapter learns from `(bm25_score, dense_score, query_features)` → `helpful?`

In [12]:
from online_learning import RiverHybridAdapter

adapter = RiverHybridAdapter()
print(f"River adapter initialized.")
print(f"  ADWIN delta: {adapter.adwin.delta}")
print(f"  Initial accuracy: {adapter.current_accuracy}")
print(f"  Total steps: {adapter.total_steps}")

River adapter initialized.
  ADWIN delta: 0.002
  Initial accuracy: 0.0
  Total steps: 0


In [13]:
# Simulate a stream of feedback using the GraphRAG response scores
import random
random.seed(42)

simulated_feedback = [
    {"query": "attention mechanism", "bm25": 12.5, "dense": 0.87, "helpful": 1},
    {"query": "BERT pre-training", "bm25": 8.3, "dense": 0.91, "helpful": 1},
    {"query": "image classification CNN", "bm25": 3.1, "dense": 0.45, "helpful": 0},
    {"query": "transformer architecture", "bm25": 15.2, "dense": 0.93, "helpful": 1},
    {"query": "reinforcement learning policy", "bm25": 1.0, "dense": 0.32, "helpful": 0},
    {"query": "language model fine-tuning", "bm25": 10.7, "dense": 0.85, "helpful": 1},
    {"query": "knowledge graph embedding", "bm25": 6.2, "dense": 0.71, "helpful": 1},
    {"query": "retrieval augmented generation", "bm25": 14.1, "dense": 0.89, "helpful": 1},
    {"query": "robot manipulation", "bm25": 0.5, "dense": 0.21, "helpful": 0},
    {"query": "multi-head attention", "bm25": 13.8, "dense": 0.92, "helpful": 1},
]

print(f"{'Step':>4} {'Query':<35} {'Helpful':>7} {'Accuracy':>8} {'Drift':>5}")
print("-" * 65)

for fb in simulated_feedback:
    entry = adapter.learn(
        query_text=fb["query"],
        bm25_top_score=fb["bm25"],
        dense_top_score=fb["dense"],
        helpful=fb["helpful"],
    )
    drift_mark = "***" if entry.drift_detected else ""
    print(f"{entry.step:>4} {fb['query']:<35} {fb['helpful']:>7} {entry.accuracy:>8.3f} {drift_mark:>5}")

Step Query                               Helpful Accuracy Drift
-----------------------------------------------------------------
   0 attention mechanism                       1    0.000      
   1 BERT pre-training                         1    0.500      
   2 image classification CNN                  0    0.667      
   3 transformer architecture                  1    0.750      
   4 reinforcement learning policy             0    0.800      
   5 language model fine-tuning                1    0.667      
   6 knowledge graph embedding                 1    0.571      
   7 retrieval augmented generation            1    0.625      
   8 robot manipulation                        0    0.667      
   9 multi-head attention                      1    0.700      


In [14]:
summary = adapter.get_summary()
print("\nAdapter Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

# Predicted alpha for a new query
alpha = adapter.predict_alpha("attention mechanism", bm25_top_score=12.0, dense_top_score=0.88)
print(f"\nPredicted alpha for 'attention mechanism': {alpha:.4f}")
print(f"  (higher = favor BM25, lower = favor dense)")


Adapter Summary:
  total_steps: 10
  current_accuracy: 0.7
  n_drifts: 0
  last_drift_step: -1

Predicted alpha for 'attention mechanism': 0.5107
  (higher = favor BM25, lower = favor dense)


---
## 6. Evaluation & Ablation Harness

The `evaluate.py` script tests the pipeline in three ablation modes:

| Mode | Description |
|------|-------------|
| `vector_only` | BM25 + Dense hybrid search, no graph filtering |
| `graph_only` | Only graph-filtered chunks, no fallback |
| `hybrid` | Full GraphRAG pipeline (graph filter with fallback) |

Metrics: **p95 latency**, **Faithfulness** (citation verification), **Answer-Relevance** (keyword overlap).

In [15]:
from evaluate import load_queries

queries = load_queries("eval_queries.csv")
print(f"Loaded {len(queries)} evaluation queries:\n")
for i, q in enumerate(queries, 1):
    kw = q.get('expected_keywords', [])
    kw_str = ', '.join(kw) if kw else '(none)'
    print(f"  [{i}] {q['query']}")
    print(f"      keywords: {kw_str}")

Loaded 5 evaluation queries:

  [1] How does the attention mechanism work in transformers?
      keywords: attention, self-attention, multi-head, scaled dot-product
  [2] What is BERT and how is it pre-trained?
      keywords: masked language model, pre-training, bidirectional, BERT
  [3] How does retrieval-augmented generation improve LLM accuracy?
      keywords: retrieval, augmented, generation, knowledge, RAG
  [4] What architecture does LLaMA use?
      keywords: LLaMA, transformer, decoder, parameters
  [5] How does LLaVA combine vision and language?
      keywords: visual, language, multimodal, instruction, LLaVA


### 6.1 Run a single mode (hybrid)

In [16]:
from evaluate import evaluate_mode, print_summary_table

# Run just the hybrid mode on 2 queries (quick demo)
demo_queries = queries[:2]
results, agg = evaluate_mode(demo_queries, mode="hybrid", top_k=5)

print_summary_table([agg])


  Evaluating mode: HYBRID
  [1/2] How does the attention mechanism work in transformers?...

2026-06-21 19:16:52,364 | INFO | Connected to Neo4j at bolt://localhost:7687


2026-06-21 19:16:52,695 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


2026-06-21 19:16:53,719 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


2026-06-21 19:16:53,723 | INFO | Cypher generated | intent: no_match


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 81.03it/s]


2026-06-21 19:17:01,544 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


2026-06-21 19:17:05,317 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


 4988ms | faith=1.00 | rel=0.75

  Mode             p95 Lat(ms)   Mean Lat    Faith  Relevance  Provenance   Chunks
------------------------------------------------------------------------------------------
  hybrid                7170.5     6136.6    1.000      0.750       1.000      5.0



### 6.2 Full ablation (all three modes)

Compare `vector_only` vs `graph_only` vs `hybrid` side by side.

In [17]:
# Run all three modes on the same queries
all_aggs = []
for mode in ["vector_only", "graph_only", "hybrid"]:
    _, agg = evaluate_mode(demo_queries, mode=mode, top_k=5)
    all_aggs.append(agg)

print_summary_table(all_aggs)


  Evaluating mode: VECTOR_ONLY
  [1/2] How does the attention mechanism work in transformers?...

2026-06-21 19:17:06,036 | INFO | Connected to Neo4j at bolt://localhost:7687


2026-06-21 19:17:06,037 | INFO | Building BM25 index from MongoDB ...


2026-06-21 19:17:06,091 | INFO | BM25 index built with 872 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

2026-06-21 19:17:29,139 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"


2026-06-21 19:17:29,147 | INFO | Retrying request to /chat/completions in 0.992551 seconds


2026-06-21 19:17:30,297 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 429 Too Many Requests"


2026-06-21 19:17:30,299 | ERROR | Answer generation failed: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 29.744730476s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'locati

 4303ms | faith=1.00 | rel=0.00

  Mode             p95 Lat(ms)   Mean Lat    Faith  Relevance  Provenance   Chunks
------------------------------------------------------------------------------------------
  vector_only           4268.1     4233.0    1.000      0.750       1.000      5.0
  graph_only            2133.7     1644.4    1.000      0.000       1.000      0.0
  hybrid                4739.4     4532.6    1.000      0.000       1.000      5.0



### 6.3 Faithfulness & Answer-Relevance scorers

In [18]:
from evaluate import compute_faithfulness, compute_answer_relevance

# Faithfulness: what fraction of citations passed provenance
faith = compute_faithfulness("some answer", verified_count=3, total_citation_count=4)
print(f"Faithfulness (3/4 verified): {faith:.2%}")

faith_perfect = compute_faithfulness("some answer", verified_count=5, total_citation_count=5)
print(f"Faithfulness (5/5 verified): {faith_perfect:.2%}")

faith_none = compute_faithfulness("some answer", verified_count=0, total_citation_count=0)
print(f"Faithfulness (no citations): {faith_none:.2%}")

# Answer-Relevance: keyword overlap
rel = compute_answer_relevance(
    query="How does attention work?",
    answer="The attention mechanism computes scaled dot-product attention.",
    expected_keywords=["attention", "self-attention", "multi-head", "scaled dot-product"],
)
print(f"\nAnswer-Relevance (keyword): {rel:.2%}")

rel_query = compute_answer_relevance(
    query="How does attention work in transformers?",
    answer="The transformer model uses multi-head attention across all layers.",
)
print(f"Answer-Relevance (query token overlap): {rel_query:.2%}")

Faithfulness (3/4 verified): 75.00%
Faithfulness (5/5 verified): 100.00%
Faithfulness (no citations): 100.00%

Answer-Relevance (keyword): 50.00%
Answer-Relevance (query token overlap): 33.33%


---
## 7. FastAPI Integration

All D3 components are exposed via two endpoints:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/graphrag` | POST | Full GraphRAG pipeline with provenance safety |
| `/feedback` | POST | Submit y/n feedback, triggers River + ADWIN |

Start the server: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`

In [19]:
import requests

BASE = "http://localhost:8000"

try:
    # Test GraphRAG endpoint
    resp = requests.post(f"{BASE}/graphrag", json={
        "query": "What papers discuss attention mechanisms?",
        "top_k": 3,
    }, timeout=60).json()

    print("POST /graphrag response:")
    print(f"  Chunks used      : {resp['chunks_used']}")
    print(f"  Graph filter     : {resp['graph_filter_applied']}")
    print(f"  Provenance score : {resp['provenance_score']}")
    print(f"  Elapsed          : {resp['elapsed_ms']:.0f}ms")
    print(f"  Answer preview   : {resp['answer'][:200]}...")

    # Test feedback endpoint with scores from GraphRAG
    fb_resp = requests.post(f"{BASE}/feedback", json={
        "query": "What papers discuss attention mechanisms?",
        "helpful": 1,
        "bm25_top_score": resp["bm25_top_score"],
        "dense_top_score": resp["dense_top_score"],
    }, timeout=10).json()

    print(f"\nPOST /feedback response:")
    print(f"  Step             : {fb_resp['step']}")
    print(f"  Accuracy         : {fb_resp['current_accuracy']}")
    print(f"  Drift detected   : {fb_resp['drift_detected']}")

except (requests.ConnectionError, requests.exceptions.JSONDecodeError):
    print("API server not running. Start it with: uvicorn app:app --port 8000")

API server not running. Start it with: uvicorn app:app --port 8000


---
## Cleanup

In [20]:
executor.close()
print("Connections closed.")
print("\nD3 notebook complete.")

Connections closed.

D3 notebook complete.
